# Handwriting generation: LSTM + Gaussian mixture (MDN) head

**Task.** Given a character, generate its pen trajectory as a sequence of steps $(dx_t, dy_t, \text{pen}_t)$.

**Pipeline in this notebook**

1. Data: `delta_data_equal_length.pkl` $\to$ padded tensors with a start token (teacher forcing)
2. Model: LSTM, a mixture head ($5K$ numbers) and a pen head (3 numbers)
3. Loss: Gaussian-mixture NLL for $(dx, dy)$ + weighted cross entropy for the pen state
4. Training with early stopping, clipping at norm 10, interruption recovery, and five-epoch previews
5. Generation: deterministic (top component mean) and sampled (with temperature)
6. Plots

**Data assumption.** `delta_data[writer][char]` is a list of arrays of shape $(T, 3)$ with columns $dx, dy, \text{pen}$. Pen states: $0$ = down, $1$ = up, $2$ = end of character.

In [ ]:
import math
import copy
from pathlib import Path
import pickle
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. Data

For one sample with targets $\mathbf{y}_0, \dots, \mathbf{y}_{T-1}$ (each $\mathbf{y}_t = (dx_t, dy_t, \text{pen}_t)$), the model input is shifted by one step:

$$
\text{input} = [\,(0,0,1),\ \mathbf{y}_0,\ \mathbf{y}_1,\ \dots,\ \mathbf{y}_{T-2}\,], \qquad
\text{target} = [\,\mathbf{y}_0,\ \mathbf{y}_1,\ \dots,\ \mathbf{y}_{T-1}\,]
$$

The first input $(0,0,1)$ is the **start token**. Generation must start with the same token.
Sequences are padded to the same length and a mask marks the real steps.
The artificial `(0,0,1)` token already stored at the start of a trajectory is removed from the targets and retained only as the first input. EOS remains a pen target; its coordinates are excluded from the density loss. Character IDs start at **1**.


In [ ]:
# ----------------------------------------------------
# A. Load the data and split by writer
# ----------------------------------------------------

with open("delta_data_equal_length.pkl", "rb") as f:
    delta_data = pickle.load(f)

# If you already have train_data / val_data / test_data, skip this split
writers = sorted(list(delta_data.keys()))
random.shuffle(writers)

n_writers = len(writers)
n_train = int(0.70 * n_writers)
n_val = int(0.15 * n_writers)

train_data = {}
val_data = {}
test_data = {}

for i in range(n_writers):
    w = writers[i]
    if i < n_train:
        train_data[w] = delta_data[w]
    elif i < n_train + n_val:
        val_data[w] = delta_data[w]
    else:
        test_data[w] = delta_data[w]

print("writers train/val/test:", len(train_data), len(val_data), len(test_data))

# ----------------------------------------------------
# B. Character ids (one unique number per character)
# ----------------------------------------------------

all_chars = []
for writer_id in delta_data:
    for char in delta_data[writer_id]:
        if char not in all_chars:
            all_chars.append(char)

# Shared one-based IDs: 1 through num_chars.
char_to_id = {char: i for i, char in enumerate(all_chars, start=1)}

num_chars = len(all_chars)
print("Total unique characters:", num_chars)

In [ ]:
# ----------------------------------------------------
# C. Build input / target lists
# ----------------------------------------------------

def build_dataset_lists(data_split):
    c_list = []
    X_list = []
    y_list = []

    for writer_id in data_split:
        for char in data_split[writer_id]:
            c_id = char_to_id[char]

            for rep in data_split[writer_id][char]:
                traj = np.array(rep, dtype=np.float32, copy=True)
                start = np.array([[0.0, 0.0, 1.0]], dtype=np.float32)
                if traj.ndim != 2 or traj.shape[1] != 3:
                    raise ValueError("Each trajectory must have shape (T, 3).")
                if len(traj) and np.array_equal(traj[0], start[0]):
                    traj = traj[1:]  # Stored start token is an input, not a target.
                if not len(traj) or traj[-1, 2] != 2:
                    raise ValueError("Each trajectory must end with EOS (pen=2).")
                if not np.isfinite(traj).all() or not np.isin(traj[:, 2], [0, 1, 2]).all():
                    raise ValueError("Trajectory values must be finite with pen states 0/1/2.")

                inp = np.concatenate([start, traj[:-1]], axis=0)
                tgt = traj

                c_list.append(c_id)
                X_list.append(inp)
                y_list.append(tgt)

    return c_list, X_list, y_list


# ----------------------------------------------------
# D. Pad to the same length and build the mask
# ----------------------------------------------------

def pad_sequences(X_list, y_list, max_len):
    n = len(X_list)

    X = np.zeros((n, max_len, 3), dtype=np.float32)
    Y = np.zeros((n, max_len, 3), dtype=np.float32)
    M = np.zeros((n, max_len), dtype=np.float32)

    for i in range(n):
        L = len(y_list[i])
        X[i, :L, :] = X_list[i]
        Y[i, :L, :] = y_list[i]
        M[i, :L] = 1.0

    return X, Y, M


c_train, X_train, y_train = build_dataset_lists(train_data)
c_val,   X_val,   y_val   = build_dataset_lists(val_data)
c_test,  X_test,  y_test  = build_dataset_lists(test_data)

max_len = 0
for seq in y_train + y_val + y_test:
    if len(seq) > max_len:
        max_len = len(seq)

print("samples train/val/test:", len(c_train), len(c_val), len(c_test))
print("longest sequence:", max_len)


def make_dataset(c_list, X_list, y_list):
    X, Y, M = pad_sequences(X_list, y_list, max_len)

    # tensor order: 0 = input, 1 = char id, 2 = target, 3 = mask
    dataset = TensorDataset(
        torch.tensor(X),
        torch.tensor(c_list, dtype=torch.long),
        torch.tensor(Y),
        torch.tensor(M)
    )
    return dataset


train_dataset = make_dataset(c_train, X_train, y_train)
val_dataset   = make_dataset(c_val,   X_val,   y_val)
test_dataset  = make_dataset(c_test,  X_test,  y_test)

BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

### Training statistics used by the loss

**Coordinate scale.** $s_d$ is the standard deviation of $dx$ and $dy$ over the real training motion steps (excluding padding and EOS). The loss divides the coordinate errors by it.

**Pen-class weights.** Pen state 0 is usually the most frequent, so the rare classes get larger weights:

$$
w_i = \left(\frac{\text{count}_{\max}}{\text{count}_i}\right)^{p}
$$

$p = 0.5$ gives the square-root weights, $p = 1$ gives the full inverse-frequency weights.

In [ ]:
# ----------------------------------------------------
# E. Coordinate std (real steps only)
# ----------------------------------------------------

train_xy = train_dataset.tensors[2][:, :, :2]            # (N, T, 2)
train_mask = train_dataset.tensors[3].bool()             # (N, T)

motion_mask = train_mask & (train_dataset.tensors[2][..., 2] != 2)
valid_xy = train_xy[motion_mask]                          # (n_valid, 2)
coord_std = valid_xy.std(dim=0, correction=0).clamp(min=1e-6).to(device)               # (2,)
print("coord_std:", coord_std)

# ----------------------------------------------------
# F. Pen-state class weights
# ----------------------------------------------------

WEIGHT_POWER = 0.5     # 0.5 = square-root weights, 1.0 = full inverse frequency

train_pen = train_dataset.tensors[2][:, :, 2].long()     # pen state
class_counts = torch.bincount(train_pen[train_mask], minlength=3).float()

pen_class_weights = (class_counts.max() / class_counts.clamp(min=1)) ** WEIGHT_POWER
pen_class_weights = pen_class_weights.to(device)

print("class counts :", class_counts)
print("class weights:", pen_class_weights)

## 2. Model

Each character ID selects a learned vector from `nn.Embedding`. Set its size with `EMBEDDING_DIM` in the training hyperparameters. IDs remain `1..num_chars`, with row `0` reserved. The vector is repeated at every time step and concatenated with `(dx, dy, pen)`, giving `3 + EMBEDDING_DIM` input features.

At every step the LSTM turns this input into a hidden vector $\mathbf{h}_t$ (the memory of the strokes so far). Two linear heads read it:

$$
\mathbf{o}_t = W\,\mathbf{h}_t + \mathbf{b} \in \mathbb{R}^{5K}, \qquad
\mathbf{s}_t = W_p\,\mathbf{h}_t + \mathbf{b}_p \in \mathbb{R}^{3}
$$

The $5K$ numbers are split as $[\,a_1..a_K \mid \mu_{1x},\mu_{1y},\dots \mid \ell_{1x},\ell_{1y},\dots\,]$:

| slice | size | meaning | becomes |
|---|---|---|---|
| mixture logits $a_k$ | $K$ | how much each Gaussian is trusted | $\pi_k = \text{softmax}(a)_k$ |
| means $\mu_{k,d}$ | $2K$ | centre of each Gaussian | used as is |
| log-stds $\ell_{k,d}$ | $2K$ | spread of each Gaussian | $\sigma_{k,d} = e^{\ell_{k,d}}$ |

The LSTM keeps the past. The mixture only describes what could come next from that past.

*If your own model class has the same `forward(x, c, hidden)` signature and returns `(xy_mdn, pen_logits, hidden)`, use it instead of this one.*

In [ ]:
class LSTMMDN(nn.Module):

    def __init__(self, num_chars, K=5, hidden_size=256, num_layers=2,
                 embedding_dim=16):
        super().__init__()

        self.num_chars = num_chars
        self.embedding_dim = embedding_dim
        # IDs are 1..num_chars; reserve row 0.
        self.char_embedding = nn.Embedding(
            num_chars + 1, embedding_dim, padding_idx=0
        )

        # Input per step: dx, dy, pen, and a learned character vector.
        self.lstm = nn.LSTM(
            input_size=3 + embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.mdn_head = nn.Linear(hidden_size, 5 * K)   # mixture parameters
        self.pen_head = nn.Linear(hidden_size, 3)       # pen-state logits

    def forward(self, x, c, hidden=None):
        # x: (B, T, 3), c: (B,) character ids

        # Select one learned vector per character and repeat it across time.
        char_ids = c.to(device=x.device, dtype=torch.long).reshape(-1)
        char_emb = self.char_embedding(char_ids)         # (B, E)
        char_emb = char_emb.unsqueeze(1).expand(-1, x.shape[1], -1)
        inp = torch.cat([x, char_emb], dim=-1)             # (B, T, 3 + E)

        h, hidden = self.lstm(inp, hidden)               # (B, T, H)

        xy_mdn = self.mdn_head(h)                        # (B, T, 5K)
        pen_logits = self.pen_head(h)                    # (B, T, 3)

        return xy_mdn, pen_logits, hidden

## 3. Loss

**Coordinates (mixture NLL).** With $z_{k,d} = \dfrac{y_d - \mu_{k,d}}{s_d\,\sigma_{k,d}}$ and independent $dx, dy$ inside each Gaussian:

$$
\log \mathcal{N}_k = -\tfrac12 \sum_{d} z_{k,d}^2 \;-\; \sum_d \ell_{k,d} \;-\; \log(2\pi)
$$

$$
\text{NLL} = -\log \sum_{k=1}^{K} \exp\!\big(\log \pi_k + \log \mathcal{N}_k\big)
$$

`logsumexp` and `log_softmax` keep this numerically stable. Dividing by $s_d$ only shifts the NLL by the constant $\log s_x + \log s_y$, so the gradients are the same as for raw units. $\mu$ stays in raw units, $\sigma$ lives in standardized units.

**Pen state (weighted cross entropy).** With true class $y$ and class weights $w$:

$$
\ell_t = -\,w_{y}\,\log p_{t,y}
$$

**Total.** Coordinate NLL is averaged over motion points $N_{motion}$ (excluding EOS and padding). Pen CE is averaged over all valid pen targets $N$, including EOS:

$$
\mathcal{L} = \frac{1}{N_{motion}}\sum_t m_t\,\mathbf{1}[y_{t,pen}\ne 2]\,\text{NLL}_t \;+\; \lambda\,\frac{1}{N}\sum_t m_t\,\ell_t
$$

with $\lambda$ = `PEN_WEIGHT`.

In [ ]:
def compute_loss_and_metrics(predictions, targets, mask, coord_std, pen_weight):
    xy_mdn, pen_logits, _ = predictions
    mask = mask.bool()
    pen_target = targets[..., 2].long()
    coord_mask = mask & (pen_target != 2)

    # A. Coordinate NLL: select motion targets BEFORE evaluating the density.
    # Padding and EOS cannot influence either coordinate loss or its gradients.
    if coord_mask.any():
        raw = xy_mdn[coord_mask]
        K = raw.shape[-1] // 5
        pi_logits, mu, log_sigma = torch.split(raw, [K, 2*K, 2*K], dim=-1)
        mu = mu.reshape(-1, K, 2)
        log_sigma = log_sigma.reshape(-1, K, 2).clamp(min=-5, max=3)
        scale = coord_std.to(device=raw.device, dtype=raw.dtype).clamp(min=1e-6)
        error = (targets[..., :2][coord_mask].unsqueeze(1) - mu) / scale
        standardized_error = error / log_sigma.exp()
        log_gaussian = (
            -0.5 * standardized_error.square().sum(dim=-1)
            -log_sigma.sum(dim=-1) - math.log(2 * math.pi)
        )
        log_pi = F.log_softmax(pi_logits, dim=-1)
        coord_loss = -torch.logsumexp(log_pi + log_gaussian, dim=-1).mean()
    else:
        coord_loss = xy_mdn[coord_mask].sum()

    # B. Weighted CE per valid pen target, including EOS.
    if mask.any():
        weights = pen_class_weights.to(device=pen_logits.device, dtype=pen_logits.dtype)
        pen_loss = F.cross_entropy(
            pen_logits[mask], pen_target[mask], weight=weights, reduction="none"
        ).mean()
    else:
        pen_loss = pen_logits[mask].sum()

    loss = coord_loss + pen_weight * pen_loss
    return loss, coord_loss, pen_loss


## 4. Training setup

* Coordinate NLL can be negative because continuous densities can exceed 1.
* Gradients are clipped to total norm **10** before each optimizer update. Recorded norms are measured before clipping; clipping does not guarantee monotonic validation loss.
* Epoch coordinate and pen losses use their respective valid-point counts, so evaluation does not depend on batching.
* Training prints both loss components and EOS recall. Previews show epoch 1 and every fifth epoch; set `PREVIEW_EVERY = 0` to disable them.
* Interrupting the training cell restores the best completed validation model and saves its state dictionary. If validation has not completed once, the initial weights are restored. No additional preview is started after interruption.
* The generation and plotting helper cells below are defined before the training cell. Run the notebook from the top after changing character IDs or rebuilding targets.


In [ ]:
# ----------------------------------------------------
# Hyperparameters
# ----------------------------------------------------

K = 5                   # number of Gaussians
HIDDEN_SIZE = 64
NUM_LAYERS = 3
EMBEDDING_DIM = 16       # Learned character-vector size
LEARNING_RATE = 1e-3
PEN_WEIGHT = 1.0        # lambda; lower it if WEIGHT_POWER = 1.0
NUM_EPOCHS = 1000
PATIENCE = 15
CHECKPOINT_PATH = "lstm_mdn_best.pth"


GRAD_CLIP = 10.0
PREVIEW_EVERY = 5
PREVIEW_MAX_STEPS = max_len
PREVIEW_CHARACTERS = None  # All characters, or ['a', 'b', 'A', '2']
PREVIEW_SAMPLED = False   # False: top-component mean; True: sampled coordinates


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    coord_sum = pen_sum = 0.0
    n_motion = n_steps = n_correct = 0
    n_up_true = n_up_hit = n_eos_true = n_eos_hit = 0
    batch_gradient_norms = []

    with torch.set_grad_enabled(training):
        for X, c, y, mask in loader:
            X, c, y, mask = (v.to(device) for v in (X, c, y, mask))
            mask_b = mask.bool()
            if not mask_b.any():
                continue
            if training:
                optimizer.zero_grad(set_to_none=True)
            predictions = model(X, c)
            loss, coord_loss, pen_loss = compute_loss_and_metrics(
                predictions, y, mask_b, coord_std, PEN_WEIGHT
            )
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite coordinate or pen loss.")
            if training:
                loss.backward()
                # Return value is the norm BEFORE clipping; optimizer sees clipped gradients.
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), max_norm=GRAD_CLIP, error_if_nonfinite=True
                ).item()
                optimizer.step()
                gradient_norm_history.append(grad_norm)
                batch_gradient_norms.append(grad_norm)

            pen_pred = predictions[1].detach().argmax(dim=-1)
            pen_true = y[..., 2].long()
            n_valid = mask_b.sum().item()
            n_coord = (mask_b & (pen_true != 2)).sum().item()
            coord_sum += coord_loss.item() * n_coord
            pen_sum += pen_loss.item() * n_valid
            n_motion += n_coord
            n_steps += n_valid
            n_correct += ((pen_pred == pen_true) & mask_b).sum().item()
            up = (pen_true == 1) & mask_b
            eos = (pen_true == 2) & mask_b
            n_up_true += up.sum().item()
            n_up_hit += (up & (pen_pred == 1)).sum().item()
            n_eos_true += eos.sum().item()
            n_eos_hit += (eos & (pen_pred == 2)).sum().item()

    if n_steps == 0:
        raise ValueError("The loader contains no valid target points.")
    coord_loss = coord_sum / max(n_motion, 1)
    pen_loss = pen_sum / n_steps
    return {
        "loss": coord_loss + PEN_WEIGHT * pen_loss,
        "coord_loss": coord_loss,
        "pen_loss": pen_loss,
        "pen_acc": n_correct / n_steps,
        "pen_up_recall": n_up_hit / n_up_true if n_up_true else float("nan"),
        "eos_recall": n_eos_hit / n_eos_true if n_eos_true else float("nan"),
        "grad_norm": float(np.mean(batch_gradient_norms)) if batch_gradient_norms else float("nan"),
    }


## 5. Generation

Generation is autoregressive: the model output at step $t$ becomes the input at step $t+1$. The first input is the start token $(0,0,1)$, as in training. The loop stops when the model predicts pen state 2 (end of character). `max_steps` is only a safety limit.

**Deterministic version.** Take the Gaussian with the largest $\pi_k$ and use its mean:

$$
k^* = \arg\max_k \pi_k, \qquad (dx, dy) = \boldsymbol{\mu}_{k^*}
$$

**Sampled version.** Pick a component, then draw from it. The temperature $\tau$ controls the randomness:

$$
k \sim \text{softmax}(a/\tau), \qquad
y_d = \mu_{k,d} + s_d\,\sqrt{\tau}\,\sigma_{k,d}\,\epsilon_d, \quad \epsilon_d \sim \mathcal{N}(0,1)
$$

$\tau = 1$ is the learned coordinate mixture (the pen state still uses argmax), $\tau < 1$ is more conservative, $\tau \to 0$ is the deterministic version.

**Pen-up boost.** Pen-up is rare and its timing varies between writers, so its probability is often below the pen-down probability at every single step. `pen_up_boost` multiplies only that probability before the argmax. Use $1.0$ (no boost) first, and raise it only if dots and crossbars are missing.

In [ ]:
@torch.inference_mode()
def generate_trajectory(model, char_id, max_steps=500,
                        initial_input=(0.0, 0.0, 1.0), pen_up_boost=1.0):
    """Top mixture-component mean, with an argmax pen state."""
    if max_steps < 1 or pen_up_boost <= 0:
        raise ValueError("max_steps and pen_up_boost must be positive.")
    parameter = next(model.parameters())
    c = torch.tensor([char_id], dtype=torch.long, device=parameter.device)
    x_t = parameter.new_tensor(initial_input).reshape(1, 1, 3)
    hidden, trajectory = None, []
    was_training = model.training
    model.eval()
    try:
        for _ in range(max_steps):
            xy_mdn, pen_logits, hidden = model(x_t, c, hidden)
            raw = xy_mdn[0, 0]
            K = raw.shape[-1] // 5
            pi_logits, mu, _ = torch.split(raw, [K, 2*K, 2*K], dim=-1)
            delta = mu.reshape(K, 2)[pi_logits.argmax()]
            pen_scores = pen_logits[0, 0].softmax(dim=-1)
            pen_scores[1] *= pen_up_boost
            pen = pen_scores.argmax()
            if pen.item() == 2:
                trajectory.append([0.0, 0.0, 2.0])
                break
            trajectory.append([delta[0].item(), delta[1].item(), pen.item()])
            x_t = torch.cat([delta, pen.to(delta.dtype).unsqueeze(0)]).reshape(1, 1, 3)
    finally:
        model.train(was_training)
    return np.asarray(trajectory, dtype=np.float32).reshape(-1, 3)


@torch.inference_mode()
def generate_sampled_trajectory(model, char_id, max_steps=500,
                                initial_input=(0.0, 0.0, 1.0), temperature=0.7,
                                pen_up_boost=1.0, seed=None):
    """Sample coordinates from the diagonal mixture; use an argmax pen state."""
    if max_steps < 1 or temperature <= 0 or pen_up_boost <= 0:
        raise ValueError("max_steps, temperature, and pen_up_boost must be positive.")
    parameter = next(model.parameters())
    c = torch.tensor([char_id], dtype=torch.long, device=parameter.device)
    x_t = parameter.new_tensor(initial_input).reshape(1, 1, 3)
    scale = coord_std.to(device=parameter.device, dtype=parameter.dtype)
    generator = None if seed is None else torch.Generator(device=parameter.device).manual_seed(seed)
    hidden, trajectory = None, []
    was_training = model.training
    model.eval()
    try:
        for _ in range(max_steps):
            xy_mdn, pen_logits, hidden = model(x_t, c, hidden)
            raw = xy_mdn[0, 0]
            K = raw.shape[-1] // 5
            pi_logits, mu, log_sigma = torch.split(raw, [K, 2*K, 2*K], dim=-1)
            mu = mu.reshape(K, 2)
            log_sigma = log_sigma.reshape(K, 2).clamp(min=-5, max=3)
            pen_scores = pen_logits[0, 0].softmax(dim=-1)
            pen_scores[1] *= pen_up_boost
            pen = pen_scores.argmax()
            if pen.item() == 2:
                trajectory.append([0.0, 0.0, 2.0])
                break
            pi = (pi_logits / temperature).softmax(dim=-1)
            k = torch.multinomial(pi, 1, generator=generator).item()
            eps = torch.randn(2, device=parameter.device, dtype=parameter.dtype, generator=generator)
            delta = mu[k] + scale * math.sqrt(temperature) * log_sigma[k].exp() * eps
            trajectory.append([delta[0].item(), delta[1].item(), pen.item()])
            x_t = torch.cat([delta, pen.to(delta.dtype).unsqueeze(0)]).reshape(1, 1, 3)
    finally:
        model.train(was_training)
    return np.asarray(trajectory, dtype=np.float32).reshape(-1, 3)


## 6. From deltas to strokes and plots

Positions are the running sum of the deltas, starting at $(0,0)$:

$$
x_t = x_{t-1} + dx_t, \qquad y_t = y_{t-1} + dy_t
$$

A step is drawn when its pen state is $0$ (down). Pen state $1$ (up) is an air move, and pen state $2$ ends the character.

In [ ]:
def deltas_to_strokes(trajectory):
    # pen_state = 0: pen is DOWN (draw this step)
    # pen_state = 1 or 2: pen is UP or end (do not draw, close the stroke)

    strokes = []
    current_stroke = []

    x, y = 0.0, 0.0

    for dx, dy, pen_state in trajectory:
        if pen_state == 2:
            break
        prev_x, prev_y = x, y
        x = x + dx
        y = y + dy

        if pen_state == 0:
            # first point of a stroke: include the "from" position too
            if not current_stroke:
                current_stroke.append((prev_x, prev_y))
            current_stroke.append((x, y))
        else:
            if current_stroke:
                strokes.append(current_stroke)
                current_stroke = []

    # save the last stroke if it exists
    if current_stroke:
        strokes.append(current_stroke)

    return strokes


def plot_strokes(trajectory, title=""):
    strokes = deltas_to_strokes(trajectory)

    for stroke in strokes:
        xs = []
        ys = []
        for point in stroke:
            xs.append(point[0])
            ys.append(point[1])
        plt.plot(xs, ys, marker=".", markersize=3)

    plt.gca().set_aspect("equal")
    plt.title(title)

In [ ]:
# Live previews: generation helpers above are defined before training starts.
from IPython.display import display


def plot_training_samples(model, epoch, display_handle=None, best=False):
    characters = sorted(char_to_id) if PREVIEW_CHARACTERS is None else list(PREVIEW_CHARACTERS)
    if not characters:
        return display_handle
    ncols = min(8, len(characters))
    nrows = math.ceil(len(characters) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3*nrows), squeeze=False)
    try:
        for ax, char in zip(axes.flat, characters):
            if char not in char_to_id:
                ax.set_title(f"{char!r}: unavailable")
                ax.axis("off")
                continue
            if PREVIEW_SAMPLED:
                trajectory = generate_sampled_trajectory(
                    model, char_to_id[char], max_steps=PREVIEW_MAX_STEPS,
                    seed=SEED + char_to_id[char],
                )
            else:
                trajectory = generate_trajectory(model, char_to_id[char], max_steps=PREVIEW_MAX_STEPS)
            strokes = deltas_to_strokes(trajectory)
            for stroke in strokes:
                xs, ys = zip(*stroke)
                ax.plot(xs, ys, "-o", markersize=2)
            if not strokes:
                ax.text(0.5, 0.5, "No drawing yet", ha="center", va="center", transform=ax.transAxes)
            ended = len(trajectory) > 0 and trajectory[-1, 2] == 2
            status = "EOS" if ended else "step limit"
            ax.set_title(f"{char!r}: {len(trajectory)} steps ({status})", fontsize=10)
            ax.set_aspect("equal")
            ax.grid(True, alpha=0.3)
        for ax in axes.flat[len(characters):]:
            ax.axis("off")
        label = "Best model" if best else "Current model"
        fig.suptitle(f"{label} — epoch {epoch}", fontsize=16)
        fig.tight_layout(rect=(0, 0, 1, 0.97))
        if display_handle is None:
            display_handle = display(fig, display_id=True)
        else:
            display_handle.update(fig)
        return display_handle
    finally:
        plt.close(fig)


## Run training
Run the following cell to train. Use the notebook interrupt/stop control to stop and restore the best model.


In [ ]:
# Run this cell to start a fresh training run. Interrupt it to restore the best model.
model = LSTMMDN(
    num_chars, K=K, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS,
    embedding_dim=EMBEDDING_DIM,
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0
best_state = copy.deepcopy(model.state_dict())
best_optimizer_state = copy.deepcopy(optimizer.state_dict())
best_scheduler_state = copy.deepcopy(scheduler.state_dict())
train_losses, val_losses = [], []
gradient_norm_history = []
history = {f"{split}_{metric}": [] for split in ("train", "val")
           for metric in ("loss", "coord_loss", "pen_loss", "eos_recall", "pen_up_recall")}
history["lr"] = []
history["train_grad_norm"] = []
_preview_handle = None
training_interrupted = False


def save_best_checkpoint():
    checkpoint = Path(CHECKPOINT_PATH)
    temporary = checkpoint.with_suffix(checkpoint.suffix + ".tmp")
    torch.save(best_state, temporary)  # Preserve the notebook's state-dict file format.
    temporary.replace(checkpoint)


try:
    for epoch in range(1, NUM_EPOCHS + 1):
        train_res = run_epoch(model, train_loader, optimizer)
        val_res = run_epoch(model, val_loader)
        scheduler.step(val_res["loss"])
        train_losses.append(train_res["loss"])
        val_losses.append(val_res["loss"])
        for split, result in (("train", train_res), ("val", val_res)):
            for metric in ("loss", "coord_loss", "pen_loss", "eos_recall", "pen_up_recall"):
                history[f"{split}_{metric}"].append(result[metric])
        history["lr"].append(optimizer.param_groups[0]["lr"])
        history["train_grad_norm"].append(train_res["grad_norm"])

        improved = val_res["loss"] < best_val_loss
        if improved:
            best_state = copy.deepcopy(model.state_dict())
            best_optimizer_state = copy.deepcopy(optimizer.state_dict())
            best_scheduler_state = copy.deepcopy(scheduler.state_dict())
            best_val_loss = val_res["loss"]
            best_epoch = epoch
            epochs_without_improvement = 0
            save_best_checkpoint()
        else:
            epochs_without_improvement += 1

        lr = optimizer.param_groups[0]["lr"]
        print(
            f"Epoch {epoch:3d} | Train: {train_res['loss']:.5f} "
            f"(Coord NLL: {train_res['coord_loss']:.5f}, Pen: {train_res['pen_loss']:.5f}) | "
            f"Val: {val_res['loss']:.5f} "
            f"(Coord NLL: {val_res['coord_loss']:.5f}, Pen: {val_res['pen_loss']:.5f}) | "
            f"EOS recall train/val: {train_res['eos_recall']:.3f}/{val_res['eos_recall']:.3f} | "
            f"Val pen-up recall: {val_res['pen_up_recall']:.3f} | LR: {lr:.1e}"
            + (" * best" if improved else "")
        )
        if PREVIEW_EVERY > 0 and (epoch == 1 or epoch % PREVIEW_EVERY == 0):
            _preview_handle = plot_training_samples(model, epoch, _preview_handle)
        if epochs_without_improvement >= PATIENCE:
            print("Early stopping.")
            break
except KeyboardInterrupt:
    training_interrupted = True
    print("\nTraining interrupted. Restoring the best completed validation checkpoint.")
finally:
    model.load_state_dict(best_state)
    optimizer.load_state_dict(best_optimizer_state)
    scheduler.load_state_dict(best_scheduler_state)
    optimizer.zero_grad(set_to_none=True)
    model.eval()
    if best_epoch > 0:
        save_best_checkpoint()
        print(f"Best epoch: {best_epoch} | Best validation loss: {best_val_loss:.5f}")
        print(f"Saved: {CHECKPOINT_PATH}")
    else:
        print("No validation epoch completed; restored the initial model weights.")

if not training_interrupted and PREVIEW_EVERY > 0 and best_epoch > 0:
    try:
        _preview_handle = plot_training_samples(model, best_epoch, _preview_handle, best=True)
    except KeyboardInterrupt:
        print("Preview interrupted; the best model is already restored and saved.")


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="validation")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.grid(True)
plt.show()

test_res = run_epoch(model, test_loader)
print("Test loss:", round(test_res["loss"], 5),
      "| coord NLL:", round(test_res["coord_loss"], 5),
      "| pen acc:", round(test_res["pen_acc"], 3),
      "| pen-up recall:", round(test_res["pen_up_recall"], 3))

In [ ]:
# ----------------------------------------------------
# One character: real sample, deterministic, sampled
# ----------------------------------------------------

char = "A"
char_id = char_to_id[char]

# first real training sample of this character
real_index = 0
for i in range(len(train_dataset)):
    if train_dataset.tensors[1][i].item() == char_id:
        real_index = i
        break

real_len = int(train_dataset.tensors[3][real_index].sum().item())
real = train_dataset.tensors[2][real_index][:real_len].numpy()

det = generate_trajectory(model, char_id, max_steps=max_len)
smp = generate_sampled_trajectory(model, char_id, max_steps=max_len, temperature=0.7)

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plot_strokes(real, "Real (" + str(len(real)) + " steps)")

plt.subplot(1, 3, 2)
plot_strokes(det, "Deterministic (" + str(len(det)) + " steps)")

plt.subplot(1, 3, 3)
plot_strokes(smp, "Sampled, T=0.7 (" + str(len(smp)) + " steps)")

plt.show()

print("pen counts (deterministic):", np.bincount(det[:, 2].astype(int), minlength=3))

In [ ]:
# ----------------------------------------------------
# All characters (deterministic), title = generated length
# ----------------------------------------------------

ncols = 8
nrows = math.ceil(num_chars / ncols)

plt.figure(figsize=(2 * ncols, 2 * nrows))

for i in range(num_chars):
    ch = all_chars[i]
    traj = generate_trajectory(model, char_to_id[ch], max_steps=max_len)

    plt.subplot(nrows, ncols, i + 1)
    plot_strokes(traj, "'" + ch + "' (" + str(len(traj)) + ")")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 7. Checks when something looks wrong

| symptom | what to look at |
|---|---|
| every character is flipped or shifted | the start token and the data orientation (must match training) |
| dots and crossbars are missing | raise `pen_up_boost` (try 5 to 20), or compare the number of pen-up steps with a real sample |
| characters stop too early | check how often the model predicts class 2, and the class weights |
| right shape but wrong size or drift at the end | error accumulates through the sum of deltas, try the sampled version with $\tau \approx 0.5$ to $0.8$ |
| two characters look alike | check the character-ID mapping and learned embeddings; compare settings of `EMBEDDING_DIM` |
| validation loss goes up while train loss goes down | overfitting, stop earlier or add dropout |